In [1]:
# Étape 1 : Charger et explorer 
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["prix"] = data.target

print(df.shape)
print(df.head())
print(df.describe())
print(df.isnull().sum())

(20640, 9)
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude   prix  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%  

In [ ]:
#Étape 2 — Nettoyage des outliers
#On va supprimer les lignes aberrantes 
# Supprimer les outliers évidents
df = df[df["AveRooms"] < 50]
df = df[df["AveOccup"] < 10]
df = df[df["AveBedrms"] < 10]

print(df.shape)  # combien de lignes restent ? On a supprimé 47 lignes aberrantes 

(20593, 9)


In [3]:
#Étape 3 — Préparer X et y
X = df.drop(columns=["prix"])
y = df["prix"]

print(X.shape)
print(y.shape)

(20593, 8)
(20593,)


In [4]:
#Étape 4 — Comparer les modèles avec cross-validation
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

modeles = {
    "Régression linéaire": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor())
    ])
}

for nom, modele in modeles.items():
    scores = cross_val_score(modele, X, y, cv=5, scoring="r2")
    print(f"{nom} → R² moyen : {scores.mean():.2f} | Écart-type : {scores.std():.2f}")

Régression linéaire → R² moyen : 0.61 | Écart-type : 0.05
Random Forest → R² moyen : 0.65 | Écart-type : 0.09
XGBoost → R² moyen : 0.65 | Écart-type : 0.04
KNN → R² moyen : 0.59 | Écart-type : 0.06


XGBoost gagne — même R² que Random Forest mais écart-type plus faible (0.04 vs 0.09) 

In [ ]:
#Étape 5 — GridSearchCV sur XGBoost
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.3]
}
#learning_rate — c'est la vitesse d'apprentissage de XGBoost. 
# Plus c'est petit, plus le modèle apprend lentement mais précisément. 0.1 > 0.3 ici !

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1  # utilise tous les CPU → plus rapide !
)

grid_search.fit(X, y)

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur R² : {grid_search.best_score_:.2f}")

Meilleurs paramètres : {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Meilleur R² : 0.69


In [6]:
#Étape 6 — Évaluation finale sur le test set
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE final : {rmse:.2f}")
print(f"R² final : {r2:.2f}")

RMSE final : 0.49
R² final : 0.82


In [7]:
import joblib

joblib.dump(best_model, r"D:\DATA_FULL_STACK\Data\model_california.pkl")
print("Modèle sauvegardé !")

# Pour recharger plus tard
model = joblib.load(r"D:\DATA_FULL_STACK\Data\model_california.pkl")
print("Modèle rechargé !")

Modèle sauvegardé !
Modèle rechargé !
